In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('events.csv')

In [3]:
df.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [4]:
df.shape

(2756101, 5)

In [5]:
visitors=set()

In [6]:
for i in range(df.shape[0]):
    visitors.add(df.iloc[i]['visitorid'])


In [7]:
len(visitors)

1407580

In [8]:
df['timestamp']=pd.to_datetime(
    df['timestamp'],
    unit='ms'
)

In [9]:
df=df.sort_values(
    ['visitorid','timestamp']
).reset_index(drop=True)

In [10]:
df['previsous_timestamp']=df.groupby('visitorid')['timestamp'].shift(1)

In [11]:
df['time_gap_min']=(df['timestamp']-df['previsous_timestamp']).dt.total_seconds()/60

In [12]:
df.head()

,timestamp,visitorid,event,itemid,transactionid,previsous_timestamp,time_gap_min
0,2015-09-11 20:49:49.439,0,view,285930,NaN,NaT,NaN
1,2015-09-11 20:52:39.591,0,view,357564,NaN,2015-09-11 20:49:49.439,2.835867
2,2015-09-11 20:55:17.175,0,view,67045,NaN,2015-09-11 20:52:39.591,2.626400
3,2015-08-13 17:46:06.444,1,view,72028,NaN,NaT,NaN
4,2015-08-07 17:51:44.567,2,view,325215,NaN,NaT,NaN


### if a visitor has been inactive for more than 30 min , start a new session because we want to train our model for window for which visitor is active and so we assume if the visitor is not active for 30min we shoud consider a new session once he is active

#### the session can be grater than 30min but the gap between the session should be less than 30min

#### finding whether the session is new or not

In [13]:
df['new_session']=(df['previsous_timestamp'].isna() | (df['time_gap_min']>30) ).astype(int)

#### identifing session by the session number

In [14]:
df

,timestamp,visitorid,event,itemid,transactionid,previsous_timestamp,time_gap_min,new_session
0,2015-09-11 20:49:49.439,0,view,285930,NaN,NaT,NaN,1
1,2015-09-11 20:52:39.591,0,view,357564,NaN,2015-09-11 20:49:49.439,2.835867,0
2,2015-09-11 20:55:17.175,0,view,67045,NaN,2015-09-11 20:52:39.591,2.626400,0
3,2015-08-13 17:46:06.444,1,view,72028,NaN,NaT,NaN,1
4,2015-08-07 17:51:44.567,2,view,325215,NaN,NaT,NaN,1
...,...,...,...,...,...,...,...,...
2756096,2015-06-10 21:46:08.922,1407575,view,121220,NaN,NaT,NaN,1
2756097,2015-06-03 15:01:29.991,1407576,view,356208,NaN,NaT,NaN,1
2756098,2015-05-17 21:48:04.867,1407577,view,427784,NaN,NaT,NaN,1
2756099,2015-05-17 01:21:23.288,1407578,view,188736,NaN,NaT,NaN,1


In [15]:
df['session_number']=(df.groupby('visitorid')['new_session'].cumsum())

In [16]:
df

,timestamp,visitorid,event,itemid,transactionid,previsous_timestamp,time_gap_min,new_session,session_number
0,2015-09-11 20:49:49.439,0,view,285930,NaN,NaT,NaN,1,1
1,2015-09-11 20:52:39.591,0,view,357564,NaN,2015-09-11 20:49:49.439,2.835867,0,1
2,2015-09-11 20:55:17.175,0,view,67045,NaN,2015-09-11 20:52:39.591,2.626400,0,1
3,2015-08-13 17:46:06.444,1,view,72028,NaN,NaT,NaN,1,1
4,2015-08-07 17:51:44.567,2,view,325215,NaN,NaT,NaN,1,1
...,...,...,...,...,...,...,...,...,...
2756096,2015-06-10 21:46:08.922,1407575,view,121220,NaN,NaT,NaN,1,1
2756097,2015-06-03 15:01:29.991,1407576,view,356208,NaN,NaT,NaN,1,1
2756098,2015-05-17 21:48:04.867,1407577,view,427784,NaN,NaT,NaN,1,1
2756099,2015-05-17 01:21:23.288,1407578,view,188736,NaN,NaT,NaN,1,1


#### providing session id for each session

In [80]:
df['session_id']=(df['visitorid'].astype(str)+'_'+ df['session_number'].astype(str))

In [105]:
df['session_id'].value_counts()

session_id
310759_9      417
1313381_27    330
519839_2      320
530033_1      310
79627_192     275
             ... 
657052_2        1
657052_1        1
180535_1        1
180536_1        1
1407579_1       1
Name: count, Length: 1761675, dtype: int64

In [106]:
session_features= (
    df.groupby('session_id')
    .agg(
        
        visitorid=('visitorid','first'),
        session_start=('timestamp','min'),
        session_end=('timestamp','max'),
        views=('event',lambda x:(x=='view').sum()),
        add_to_cart=('event',lambda x:(x=='addtocart').sum()),
        unique_item_viewed=(
            'itemid',
            lambda x : x[df.loc[x.index,'event'].eq('view')].nunique()
        ),
        transaction_count=('event', lambda x : (x=='transaction').sum())                 
                   
    )
    .reset_index()
)

In [107]:
session_features

,session_id,visitorid,session_start,session_end,views,add_to_cart,unique_item_viewed,transaction_count
0,0_1,0,2015-09-11 20:49:49.439,2015-09-11 20:55:17.175,3,0,3,0
1,1000000_1,1000000,2015-06-05 18:16:10.629,2015-06-05 18:16:10.629,1,0,1,0
2,1000001_1,1000001,2015-07-07 18:12:14.953,2015-07-07 18:12:14.953,1,0,1,0
3,1000001_2,1000001,2015-07-24 20:18:15.303,2015-07-24 20:35:57.029,3,0,3,0
4,1000001_3,1000001,2015-07-29 20:38:29.170,2015-07-29 20:38:29.170,1,0,1,0
...,...,...,...,...,...,...,...,...
1761670,9999_1,9999,2015-05-14 23:22:55.338,2015-05-14 23:22:55.338,1,0,1,0
1761671,999_1,999,2015-05-29 00:35:57.524,2015-05-29 00:35:57.524,1,0,1,0
1761672,99_1,99,2015-05-10 03:23:13.472,2015-05-10 03:23:13.472,1,0,1,0
1761673,99_2,99,2015-05-10 18:30:31.676,2015-05-10 18:30:31.676,1,0,1,0


In [108]:
session_features['session_duration_min']=((session_features['session_end']-session_features['session_start']).dt.total_seconds()/60)

In [109]:
session_features['target']=(session_features['transaction_count']>0 ).astype(int)

In [110]:
session_features['target'].value_counts()

target
0    1747378
1      14297
Name: count, dtype: int64

In [111]:
session_features['month']=(session_features['session_start'].dt.month)
session_features['day']=(session_features['session_start'].dt.dayofweek)
session_features['hour']=(session_features['session_start'].dt.hour)

In [112]:
session_features.head()

,session_id,visitorid,session_start,session_end,views,add_to_cart,unique_item_viewed,transaction_count,session_duration_min,target,month,day,hour
0,0_1,0,2015-09-11 20:49:49.439,2015-09-11 20:55:17.175,3,0,3,0,5.462267,0,9,4,20
1,1000000_1,1000000,2015-06-05 18:16:10.629,2015-06-05 18:16:10.629,1,0,1,0,0.000000,0,6,4,18
2,1000001_1,1000001,2015-07-07 18:12:14.953,2015-07-07 18:12:14.953,1,0,1,0,0.000000,0,7,1,18
3,1000001_2,1000001,2015-07-24 20:18:15.303,2015-07-24 20:35:57.029,3,0,3,0,17.695433,0,7,4,20
4,1000001_3,1000001,2015-07-29 20:38:29.170,2015-07-29 20:38:29.170,1,0,1,0,0.000000,0,7,2,20


In [113]:
model_data=session_features.drop(
    columns=['target','transaction_count','session_start','session_end','session_id','visitorid']
)

In [114]:
target_data=session_features['target']

In [115]:
model_data.head()

,views,add_to_cart,unique_item_viewed,session_duration_min,month,day,hour
0,3,0,3,5.462267,9,4,20
1,1,0,1,0.000000,6,4,18
2,1,0,1,0.000000,7,1,18
3,3,0,3,17.695433,7,4,20
4,1,0,1,0.000000,7,2,20


In [116]:
target_data.value_counts()

target
0    1747378
1      14297
Name: count, dtype: int64

In [117]:
model_data.to_csv('data_for_model_traning.csv')
target_data.to_csv('target_data.csv')

In [118]:
session_features.shape

(1761675, 13)

In [119]:
max=0
for i in range(session_features.shape[0]): 
    if session_features.iloc[i]['session_duration_min'] >= max:
        max=session_features.iloc[i]['session_duration_min']
        # print(i)
        # print(session_features.iloc[i])


In [120]:
max

np.float64(728.4041333333333)

In [ ]:
model_data